# 🛠️ Phase 1: Core Matching Engine

In [1]:
pip install geopy


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time

In [3]:
# Initialize Geolocator
geolocator = Nominatim(user_agent="tactic_sense_locator", timeout=10)
geo_cache = {}

def get_lat_lon(city, country):
    city = str(city).strip()
    country = str(country).strip()
    key = (city.lower(), country.lower())
    
    if not city or not country or pd.isna(city) or pd.isna(country):
        return np.nan, np.nan
    
    if key in geo_cache:
        return geo_cache[key]
    
    try:
        location = geolocator.geocode(f"{city}, {country}")
        time.sleep(1.2)  # polite delay
        if location:
            coords = (location.latitude, location.longitude)
            geo_cache[key] = coords
            return coords
        else:
            return np.nan, np.nan
    except Exception as e:
        print(f"Error geolocating {city}, {country}: {e}")
        return np.nan, np.nan


In [4]:
# Define file paths (for your system adjust the paths if needed)
# Correct file paths on your machine
file_paths = {
    "clubs": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\clubs_dataset.csv",
    "communication_boxes": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\communication_boxes_dataset.csv",
    "equipment_suppliers": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\equipment_suppliers_dataset.csv",
    "fitness_clubs": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\fitness_clubs_dataset.xlsx",
    "player_agents": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\player_agent_dataset.csv",
    "players": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\players_dataset.csv",
    "recruiting_agents": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\recruiting_agents_dataset.csv",
    "service_providers": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\service_providers_dataset.csv",
    "sponsors": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\sponsors_dataset.csv",
    "sports_clothing_brands": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\sports_clothing_brands_dataset.csv",
    "sports_management_agencies": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\sports_management_agencies_dataset.csv",
    "travel_agencies": r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\travel_agencies_dataset.csv",
}

In [5]:
# Function to load each dataset properly
def load_dataset(name, path):
    if path.endswith('.csv'):
        df = pd.read_csv(path, encoding='latin1')  # <-- Fixed here
    elif path.endswith('.xlsx'):
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type for {path}")
    df['source_dataset'] = name
    return df

In [6]:
# Load all datasets
datasets = {}
for name, path in file_paths.items():
    datasets[name] = load_dataset(name, path)

# Simple preview
for key, df in datasets.items():
    print(f"{key.upper()} SAMPLE:\n", df.head(2), "\n\n")

CLUBS SAMPLE:
    ranking     club name country           league                 stadium  \
0      265  1o de Agosto  Angola         Girabola  Estadio 11 de Novembro   
1     2530    1o de Maio  Angola  Segunda Divisao  Estadio 11 de Novembro   

  formation_preference            playing_style  market_budget  salary_budget  \
0            '3-4-2-1'  North African Technical   3.634855e+09   1.453942e+09   
1              '4-5-1'               High Press   9.540010e+08   3.816004e+08   

                                    transfer_history  avg_transfer_fee_paid  \
0  {"last_5_transfers": [{"year": 2021, "player":...          523560.538980   
1  {"last_5_transfers": [{"year": 2020, "player":...          269316.382051   

  avg_contract_length  verified_status recruitment_priority   scouting_focus  \
0             3 years             True    Academy graduates     Asian market   
1             2 years             True      Young prospects  African talents   

   profile_rating             

In [7]:
# Creating the universal pool
pool = []

# Helper to normalize entries
def create_entry(row, stakeholder_type, defaults):
    lat, lon = row.get('latitude', np.nan), row.get('longitude', np.nan)
    if pd.isna(lat) or pd.isna(lon):
        lat, lon = get_lat_lon(row.get('city', ''), row.get('country', ''))
        time.sleep(1)  # polite delay to not overload geopy servers

    return {
        "id": row.get('id', row.index),
        "name": row.get('name', row.get('company_name', row.get('full_name', 'Unknown'))),
        "type": stakeholder_type,
        "city": row.get('city', 'Unknown'),
        "country": row.get('country', 'Unknown'),
        "latitude": lat,
        "longitude": lon,
        "industry": row.get('industry', defaults.get('industry', 'Unknown')),
        "price_level": row.get('price_range', defaults.get('price_level', 'Unknown')),
        "verified": row.get('verified_status', False),
        "rating": row.get('profile_rating', defaults.get('rating', 3.0)),
    }

In [8]:
# Specify default values per dataset
stakeholder_defaults = {
    'clubs': {"industry": "Football Club", "price_level": "High", "rating": 4.0},
    'players': {"industry": "Football Player", "price_level": "Medium", "rating": 3.5},
    'sponsors': {"industry": "Sponsor", "price_level": "High", "rating": 4.5},
    'service_providers': {"industry": "Service Provider", "price_level": "Medium", "rating": 4.0},
    'travel_agencies': {"industry": "Travel Agency", "price_level": "Medium", "rating": 4.0},
    'communication_boxes': {"industry": "Communication Agency", "price_level": "Medium", "rating": 3.8},
    'equipment_suppliers': {"industry": "Equipment Supplier", "price_level": "Medium", "rating": 4.2},
    'sports_clothing_brands': {"industry": "Sportswear Brand", "price_level": "High", "rating": 4.5},
    'sports_management_agencies': {"industry": "Sports Management Agency", "price_level": "High", "rating": 4.3},
    'recruiting_agents': {"industry": "Recruiting Agent", "price_level": "Medium", "rating": 4.1},
    'player_agents': {"industry": "Player Agent", "price_level": "Medium", "rating": 4.1},
    'fitness_clubs': {"industry": "Fitness Club", "price_level": "Medium", "rating": 3.9},
}

In [9]:
pool = []  # initialize the pool

counter = 0  # counter to track progress
total_rows = sum(len(df) for df in datasets.values())  # total expected rows

for key, df in datasets.items():
    for idx, row in df.iterrows():
        lat = row.get('latitude', np.nan)
        lon = row.get('longitude', np.nan)
        city = row.get('city', '')
        country = row.get('country', '')

        # Skip geolocation if lat/lon are already available
        if not pd.isna(lat) and not pd.isna(lon):
            coords = (lat, lon)
        else:
            # Skip geolocation if city/country missing
            if not city or not country or pd.isna(city) or pd.isna(country):
                coords = (np.nan, np.nan)
            else:
                coords = get_lat_lon(city, country)

        entry = {
            "id": row.get('id', idx),
            "name": row.get('name', row.get('company_name', row.get('full_name', 'Unknown'))),
            "type": key,
            "city": city if city else "Unknown",
            "country": country if country else "Unknown",
            "latitude": coords[0],
            "longitude": coords[1],
            "industry": row.get('industry', 'Unknown'),
            "price_level": row.get('price_range', 'Unknown'),
            "verified": row.get('verified_status', False),
            "rating": row.get('profile_rating', 3.0),
        }

        pool.append(entry)

        # Show progress every 50 entries
        counter += 1
        if counter % 50 == 0:
            print(f"Processed {counter}/{total_rows} entries...")

# Convert pool to a DataFrame
universal_pool = pd.DataFrame(pool)
print("✅ Universal pool created successfully! Total entries:", len(universal_pool))


Processed 50/7439 entries...
Processed 100/7439 entries...
Processed 150/7439 entries...
Processed 200/7439 entries...
Processed 250/7439 entries...
Processed 300/7439 entries...
Processed 350/7439 entries...
Processed 400/7439 entries...
Processed 450/7439 entries...
Processed 500/7439 entries...
Processed 550/7439 entries...
Processed 600/7439 entries...
Processed 650/7439 entries...
Processed 700/7439 entries...
Processed 750/7439 entries...
Processed 800/7439 entries...
Processed 850/7439 entries...
Processed 900/7439 entries...
Processed 950/7439 entries...
Processed 1000/7439 entries...
Processed 1050/7439 entries...
Processed 1100/7439 entries...
Processed 1150/7439 entries...
Processed 1200/7439 entries...
Processed 1250/7439 entries...
Processed 1300/7439 entries...
Processed 1350/7439 entries...
Processed 1400/7439 entries...
Processed 1450/7439 entries...
Processed 1500/7439 entries...
Processed 1550/7439 entries...
Processed 1600/7439 entries...
Processed 1650/7439 entries.

In [10]:
# Save the universal pool to CSV
universal_pool.to_csv(r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\universal_pool.csv", index=False)
print("✅ Universal pool saved successfully to CSV!")


✅ Universal pool saved successfully to CSV!


In [11]:
universal_pool = pd.read_csv(r"C:\Users\nadza\OneDrive\Desktop\Tactic_Sense_Dataset\universal_pool.csv")


# 🛠️ Phase 2: Full Matching Engine

In [14]:
from geopy.distance import geodesic

def match_stakeholders(user_location, target_type, preferences, top_n=5):
    user_lat, user_lon = get_lat_lon(user_location['city'], user_location['country'])
    if pd.isna(user_lat) or pd.isna(user_lon):
        print("❌ Could not find coordinates for user location.")
        return []
    
    # Filter only by stakeholder type
    candidates = universal_pool[(universal_pool['type'] == target_type)]
    
    results = []
    
    for idx, row in candidates.iterrows():
        if pd.isna(row['latitude']) or pd.isna(row['longitude']):
            continue

        candidate_loc = (row['latitude'], row['longitude'])
        user_loc = (user_lat, user_lon)
        distance_km = geodesic(user_loc, candidate_loc).kilometers
        
        # Distance Score (1 if 0km, drops linearly to 0 at 500km)
        distance_score = max(0, 1 - (distance_km / 500))
        
        # --- New Preference Match Calculation ---
        preference_match_score = 0
        
        # Industry match
        if preferences.get('industry') and isinstance(row['industry'], str):
            if preferences['industry'].lower() in row['industry'].lower():
                preference_match_score += 0.5
        
        # Price level match
        if preferences.get('price_level') and isinstance(row['price_level'], str):
            if preferences['price_level'].lower() == row['price_level'].lower():
                preference_match_score += 0.3
        
        # Verified match
        if preferences.get('verified_only'):
            if row['verified'] == True:
                preference_match_score += 0.2
        
        # Normalize preference match score
        preference_match_score = min(preference_match_score, 1.0)
        
        # --- Quality Score ---
        quality_score = (row['rating'] / 5) * (1.2 if row['verified'] else 1.0)
        quality_score = min(quality_score, 1.0)
        
        # --- Final Score ---
        final_score = (0.5 * preference_match_score) + (0.3 * distance_score) + (0.2 * quality_score)
        
        results.append({
            "name": row['name'],
            "city": row['city'],
            "country": row['country'],
            "distance_km": round(distance_km, 2),
            "final_score": round(final_score, 4),
            "preference_score": round(preference_match_score, 2),
            "distance_score": round(distance_score, 2),
            "quality_score": round(quality_score, 2),
            "details": f"Distance: {round(distance_km, 1)}km, Verified: {row['verified']}, Rating: {row['rating']}/5"
        })
    
    results = sorted(results, key=lambda x: x['final_score'], reverse=True)
    return results[:top_n]



In [19]:
# Your user input
user_location = {"city": "Dakar", "country": "Senegal"}
target_type = "sponsors"  # Make sure this matches one of your universal_pool['type'] values

preferences = {
    "industry": None,  # No filter on industry for now (just to test)
    "price_level": None,  # No filter on price level
    "verified_only": False,  # Allow verified and non-verified
    "min_rating": 0  # Allow all ratings
}

# Call the matching function
matches = match_stakeholders(user_location, target_type, preferences, top_n=5)

# Check and display matches
if not matches:
    print("⚠️ No matches found.")
else:
    for match in matches:
        print(f"🏆 {match['name']} | {match['city']}, {match['country']}")
        print(f"📋 {match['details']}")
        print(f"🔹 Preference Score: {match.get('preference_score', 'N/A')}, Distance Score: {match.get('distance_score', 'N/A')}, Quality Score: {match.get('quality_score', 'N/A')}")
        print(f"⭐ Final Match Score: {match['final_score']}")
        print("------")



⚠️ No matches found.


In [16]:
candidates = universal_pool[(universal_pool['type'] == target_type)]
print(f"🔎 Number of candidates found for type '{target_type}': {len(candidates)}")


🔎 Number of candidates found for type 'sponsors': 228


In [17]:
print(universal_pool['type'].unique())


['clubs' 'communication_boxes' 'equipment_suppliers' 'fitness_clubs'
 'player_agents' 'players' 'recruiting_agents' 'service_providers'
 'sponsors' 'sports_clothing_brands' 'sports_management_agencies'
 'travel_agencies']


In [18]:
print(f"User coordinates: {user_lat}, {user_lon}")


NameError: name 'user_lat' is not defined

In [ ]:
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time

In [ ]:
# Initialize Geolocator
geolocator = Nominatim(user_agent="tactic_sense_locator")

def get_lat_lon(city, country):
    try:
        location = geolocator.geocode(f"{city}, {country}")
        if location:
            return location.latitude, location.longitude
        else:
            return np.nan, np.nan
    except Exception as e:
        print(f"Error geolocating {city}, {country}: {e}")
        return np.nan, np.nan

In [ ]:
# Define file paths (adjusted to local paths)
file_paths = {
    "clubs": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\clubs_dataset.csv",
    "communication_boxes": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\communication_boxes_dataset.csv",
    "equipment_suppliers": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\equipment_suppliers_dataset.csv",
    "fitness_clubs": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\fitness_clubs_dataset.xlsx",
    "player_agents": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\player_agent_dataset.csv",
    "players": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\players_dataset.csv",
    "recruiting_agents": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\recruiting_agents_dataset.csv",
    "service_providers": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\service_providers_dataset.csv",
    "sponsors": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\sponsors_dataset.csv",
    "sports_clothing_brands": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\sports_clothing_brands_dataset.csv",
    "sports_management_agencies": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\sports_management_agencies_dataset.csv",
    "travel_agencies": r"C:\\Users\\nadza\\OneDrive\\Desktop\\Tactic_Sense_Dataset\\travel_agencies_dataset.csv"
}

In [ ]:
# Function to load each dataset properly
def load_dataset(name, path):
    if path.endswith('.csv'):
        df = pd.read_csv(path, encoding='latin1')
    elif path.endswith('.xlsx'):
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type for {path}")
    df['source_dataset'] = name
    return df

In [ ]:
# Load all datasets
datasets = {}
for name, path in file_paths.items():
    datasets[name] = load_dataset(name, path)

# Creating the universal pool
pool = []

In [ ]:
# Helper to normalize entries
def create_entry(row, stakeholder_type, defaults):
    lat, lon = row.get('latitude', np.nan), row.get('longitude', np.nan)
    if pd.isna(lat) or pd.isna(lon):
        lat, lon = get_lat_lon(row.get('city', ''), row.get('country', ''))
        time.sleep(1)  # polite delay to not overload geopy servers

    return {
        "id": row.get('id', row.index),
        "name": row.get('name', row.get('company_name', row.get('full_name', 'Unknown'))),
        "type": stakeholder_type,
        "city": row.get('city', 'Unknown'),
        "country": row.get('country', 'Unknown'),
        "latitude": lat,
        "longitude": lon,
        "industry": row.get('industry', defaults.get('industry', 'Unknown')),
        "price_level": row.get('price_range', defaults.get('price_level', 'Unknown')),
        "verified": row.get('verified_status', False),
        "rating": row.get('profile_rating', defaults.get('rating', 3.0)),
    }

In [ ]:
# Specify default values per dataset
stakeholder_defaults = {
    'clubs': {"industry": "Football Club", "price_level": "High", "rating": 4.0},
    'players': {"industry": "Football Player", "price_level": "Medium", "rating": 3.5},
    'sponsors': {"industry": "Sponsor", "price_level": "High", "rating": 4.5},
    'service_providers': {"industry": "Service Provider", "price_level": "Medium", "rating": 4.0},
    'travel_agencies': {"industry": "Travel Agency", "price_level": "Medium", "rating": 4.0},
    'communication_boxes': {"industry": "Communication Agency", "price_level": "Medium", "rating": 3.8},
    'equipment_suppliers': {"industry": "Equipment Supplier", "price_level": "Medium", "rating": 4.2},
    'sports_clothing_brands': {"industry": "Sportswear Brand", "price_level": "High", "rating": 4.5},
    'sports_management_agencies': {"industry": "Sports Management Agency", "price_level": "High", "rating": 4.3},
    'recruiting_agents': {"industry": "Recruiting Agent", "price_level": "Medium", "rating": 4.1},
    'player_agents': {"industry": "Player Agent", "price_level": "Medium", "rating": 4.1},
    'fitness_clubs': {"industry": "Fitness Club", "price_level": "Medium", "rating": 3.9},
}

In [ ]:
# Populate the pool
for key, df in datasets.items():
    for idx, row in df.iterrows():
        entry = create_entry(row, key, stakeholder_defaults.get(key, {}))
        pool.append(entry)

# Convert to DataFrame
universal_pool = pd.DataFrame(pool)

In [ ]:
# MATCHING ENGINE START

# Matching engine function
def match_stakeholders(user_location, target_type, preferences, top_n=5):
    user_lat, user_lon = get_lat_lon(user_location['city'], user_location['country'])
    if pd.isna(user_lat) or pd.isna(user_lon):
        print("Could not find coordinates for user location.")
        return []

    candidates = universal_pool[(universal_pool['type'] == target_type)]


In [ ]:
    # Apply preference filtering
    if preferences.get('industry'):
        candidates = candidates[candidates['industry'].str.contains(preferences['industry'], case=False, na=False)]
    if preferences.get('price_level'):
        candidates = candidates[candidates['price_level'].str.lower() == preferences['price_level'].lower()]
    if preferences.get('verified_only'):
        candidates = candidates[candidates['verified'] == True]
    if preferences.get('min_rating'):
        candidates = candidates[candidates['rating'] >= preferences['min_rating']]

In [ ]:


    # Scoring each candidate
    results = []
    for idx, row in candidates.iterrows():
        if pd.isna(row['latitude']) or pd.isna(row['longitude']):
            continue

        candidate_loc = (row['latitude'], row['longitude'])
        user_loc = (user_lat, user_lon)
        distance_km = geodesic(user_loc, candidate_loc).kilometers

        # Distance Score
        distance_score = max(0, 1 - (distance_km / 500))  # Normalize: 0 at 500km

        # Preference Score (full match = 1)
        preference_score = 1.0

        # Quality Score
        quality_score = (row['rating'] / 5) * (1.2 if row['verified'] else 1.0)
        quality_score = min(quality_score, 1.0)

        # Final Score
        final_score = (0.5 * preference_score) + (0.3 * distance_score) + (0.2 * quality_score)

        results.append({
            "name": row['name'],
            "city": row['city'],
            "country": row['country'],
            "distance_km": round(distance_km, 2),
            "final_score": round(final_score, 4),
            "details": f"Distance: {round(distance_km,1)}km, Verified: {row['verified']}, Rating: {row['rating']}/5"
        })

    results = sorted(results, key=lambda x: x['final_score'], reverse=True)
    return results[:top_n]
